# Plan de mise en pratique DJMA

**Objectif final**

Comparer :

- une description experte de pathologie végétale (référence)

- une description générée par un modèle multimodal (GPT, Moondream)

par mon modèle DistilBERT+LoRA NLI, qui prédit : entailment, neutral ou contradiction.
Il faut aussi définir un seuil limite



In [ ]:
#!pip install deep-translator httpx

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata
import html
from pathlib import Path
from bs4 import BeautifulSoup
from deep_translator import GoogleTranslator
import time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

# Phase 1 - Data engineering

Chargement et Nettoyage des données

Construction de la référence experte

mapping expert ↔ image


## Chargement et Nettoyage des données

In [ ]:
# Charger les données
df_expert = pd.read_csv('/content/drive/MyDrive/PROJET_DJMA/bioravageurs.csv', encoding='latin1', sep=';')
df_inference = pd.read_excel('/content/drive/MyDrive/PROJET_DJMA/Inference.xlsx')

print(f"\n Références expertes : {len(df_expert)} pathologies")
print(f"Colonnes : {df_expert.columns.tolist()}")

print(f"\n Inférences modèles : {len(df_inference)} images")
print(f"Colonnes : {df_inference.columns.tolist()}")

# Aperçu
print("\n--- Exemple de référence experte ---")
print(df_expert.iloc[0])

print("\n--- Exemple d'inférence ---")
print(df_inference.iloc[0])

In [ ]:
# Supprimer toutes les colonnes Unnamed + Molmo + Observation
drop_cols = (
    [c for c in df_inference.columns if isinstance(c, str) and c.lower().startswith("unnamed")]
    + [c for c in ["Observation "] if c in df_inference.columns]
)
df_inference = df_inference.drop(columns=drop_cols)

print(f" Colonnes conservées : {df_inference.columns.tolist()}")

In [ ]:
# Nettoyage texte

def strip_html(s: str) -> str:
    """Enlève HTML proprement + decode entités."""
    if pd.isna(s):
        return ""
    s = html.unescape(str(s))
    return BeautifulSoup(s, "html.parser").get_text(" ")

def normalize_text(s: str) -> str:
    """
    Normalise ponctuation/espaces + apostrophes.
    Retire guillemets typographiques/ASCII mais garde les apostrophes.
    """
    if pd.isna(s):
        return ""
    s = str(s)

    # Normaliser apostrophes typographiques vers '
    s = re.sub(r"[’ʼ`´]", "'", s)

    # Supprimer guillemets (sans toucher à l'apostrophe)
    s = re.sub(r'[“”„‟«»"]', "", s)

    # Espaces multiples + sauts de ligne
    s = re.sub(r"\s+", " ", s).strip()

    # Pas d'espace avant ponctuation
    s = re.sub(r"\s+([.,;:!?])", r"\1", s)

    return s.strip()

def clean_text(s: str, is_html: bool = False) -> str:
    if pd.isna(s):
        return ""
    if is_html:
        s = strip_html(s)
    return normalize_text(s)

def normalize_img(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    # enlever accents
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    # espaces et séparateurs -> underscore
    s = re.sub(r"[\s\-\/]+", "_", s)
    # enlever tout ce qui n'est pas alphanum ou underscore
    s = re.sub(r"[^a-z0-9_]", "", s)
    # compacter underscores
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def parse_vf(x):
    """
    Convertit V/F (ou variantes) -> bool
    """
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s in {"V"}: return True
    if s in {"F"}: return False
    print(f" Valeur inattendue dans annotation : '{x}'")
    return None

# 1) NETTOYAGE df_inference
df_inference.columns = df_inference.columns.map(lambda x: x.strip() if isinstance(x, str) else x)

# Nettoyer textes modèles
for col in ["Moondream", "GPT-5"]:
    if col in df_inference.columns:
        df_inference[col] = df_inference[col].apply(lambda x: clean_text(x, is_html=False))

# Nettoyer nom d'image (léger, sans “inventer”)
df_inference["Image"] = (
    df_inference["Image"]
    .astype(str)
    .map(lambda s: normalize_text(s))
    .str.strip()
)

# annotations -> bool
ann_cols = ["Moondream annotation", "GPT-5 annotation"]
for c in ann_cols:
    if c in df_inference.columns:
        df_inference[c] = df_inference[c].apply(parse_vf)

print("\n✅ df_inference nettoyé | Colonnes:", df_inference.columns.tolist())


# 2) NETTOYAGE df_expert
# Nettoyage HTML + texte sur colonnes descriptives
expert_text_cols = ["description", "how_identify", "favorable_conditions"]
for col in expert_text_cols:
    if col in df_expert.columns:
        df_expert[col] = df_expert[col].apply(lambda x: clean_text(x, is_html=True))

# Nettoyage léger sur name/type
for col in ["name", "Type"]:
    if col in df_expert.columns:
        df_expert[col] = df_expert[col].apply(lambda x: clean_text(x, is_html=False))

print("✅ df_expert nettoyé | Colonnes:", df_expert.columns.tolist())

# Image normalisée
df_inference["image_key"] = df_inference["Image"].apply(normalize_img)
df_expert["name_key"] = df_expert["name"].apply(normalize_img)

print("\n Exemples clés :")
print(df_inference[["Image", "image_key"]].head())
print(df_expert[["name", "name_key"]].head())

print("\n --- Nettoyage terminé ---")

# 3) CHECKS
print("\n ---- STATISTIQUES FINALES ----")

print(f"\n RÉFÉRENCES EXPERTES (df_expert) :")
print(f"   - Pathologies : {len(df_expert)}")
print(f"   - Types : {df_expert['Type'].value_counts().to_dict()}")

print(f"\n INFÉRENCES (df_inference) :")
print(f"   - Images : {len(df_inference)}")
print(f"   - Types : {df_inference['Type'].value_counts().to_dict()}")

# AFFICHAGE FINAL
print("\n ------ VUE D'ENSEMBLE DES DONNÉES NETTOYÉES ------- ")

# Afficher un exemple complet de chaque dataset
print("\n Exemple de référence experte :")
print(df_expert.iloc[0])

print("\n Exemple d'inférence :")
print(df_inference.iloc[0])


## Construction de la référence experte

In [ ]:
print("\n -------- CONSTRUCTION DE LA RÉFÉRENCE EXPERTE (3 MODES) -------")

def build_reference(row, mode="B"):
    """
    Construit une référence experte selon 3 modes de complexité.

    Modes :
    - A (Minimal)   : description seule
    - B (Équilibré) : description + how_identify
    - C (Complet)   : description + how_identify + favorable_conditions

    Args:
        row: Ligne du DataFrame
        mode: "A", "B" ou "C"

    Returns:
        str: Référence construite
    """
    # Récupérer les champs (avec valeurs par défaut vides)
    desc = (row.get('description') or "").strip()
    how = (row.get('how_identify') or "").strip()
    cond = (row.get('favorable_conditions') or "").strip()

    # Sélectionner les parties selon le mode
    if mode == "A":
        parts = [desc]
    elif mode == "B":
        parts = [desc, how]
    elif mode == "C":
        parts = [desc, how, cond]
    else:
        raise ValueError(f"Mode invalide : {mode}. Utilisez 'A', 'B' ou 'C'.")

    # Filtrer les parties vides et joindre
    parts = [p for p in parts if p]

    # Joindre avec un séparateur (point + espace)
    return '. '.join(parts) if parts else ""

# Créer les 3 versions de références
print("\n Création des références selon 3 modes...")

df_expert['reference_A_fr'] = df_expert.apply(lambda row: build_reference(row, mode="A"), axis=1)
df_expert['reference_B_fr'] = df_expert.apply(lambda row: build_reference(row, mode="B"), axis=1)
df_expert['reference_C_fr'] = df_expert.apply(lambda row: build_reference(row, mode="C"), axis=1)

print(f"\n✅ {len(df_expert)} références construites (3 modes)")

# Analyse comparative des modes
print("\n --- COMPARAISON DES MODES ----")

# Longueur moyenne en mots
for mode in ['A', 'B', 'C']:
    col = f'reference_{mode}_fr'
    df_expert[f'{col}_length'] = df_expert[col].apply(lambda x: len(x.split()) if x else 0)
    avg_length = df_expert[f'{col}_length'].mean()
    min_length = df_expert[f'{col}_length'].min()
    max_length = df_expert[f'{col}_length'].max()

    print(f"\nMode {mode} :")
    print(f"  Longueur moyenne : {avg_length:.1f} mots")
    print(f"  Min / Max : {min_length} / {max_length} mots")

# Vérification : Exemples de chaque mode
print("\n ---- EXEMPLES DE RÉFÉRENCES (3 modes) ----- ")
for i in range(min(2, len(df_expert))):
    print(f"\n{'='*60}")
    print(f"{df_expert.iloc[i]['name']} ({df_expert.iloc[i]['Type']})")
    print(f"{'='*60}")

    print(f"\n📌 MODE A (Minimal - description seule) :")
    print(f"   {df_expert.iloc[i]['reference_A_fr'][:300]}...")

    print(f"\n📌 MODE B (Équilibré - description + identification) :")
    print(f"   {df_expert.iloc[i]['reference_B_fr'][:300]}...")

    print(f"\n📌 MODE C (Complet - tout) :")
    print(f"   {df_expert.iloc[i]['reference_C_fr'][:300]}...")


In [ ]:
# Sélectionner le mode par défaut
DEFAULT_MODE = "B"
df_expert['reference_fr'] = df_expert[f'reference_{DEFAULT_MODE}_fr'].copy()

print(f"\n✅ Mode {DEFAULT_MODE} sélectionné comme référence par défaut")
print(f"   Colonne utilisée : 'reference_fr' (= reference_{DEFAULT_MODE}_fr)")

## MAPPING


In [ ]:
# Afficher la colonne name de expert et Image de Inference
print("Noms des pathologies dans df_expert:")
print(df_expert['name'].unique())

print("\nNoms des images dans df_inference:")
print(df_inference['Image'].unique())

print("\n ",df_expert.iloc[0])

print("\n ",df_inference.iloc[0])

In [ ]:
print("\n --- CRÉATION DU MAPPING IMAGE → PATHOLOGIE ---- ")

# MAPPING EXPERT - IMAGES → PATHOLOGIES (BASE EXPERTE)
image_to_pathology = {
    # CARENCES / TROUBLES PHYSIOLOGIQUES
    'cul_noir': 'Cul noir',
    'chlorose': 'Chlorose',

    # RAVAGEURS (INSECTES & ACARIENS)
    'aleurodes': 'Aleurode',
    'tetranyque': 'Araignée rouge',
    'pieride_du_chou': 'Piéride du chou',
    'puceron': 'Pucerons',
    'tuta_absoluta_tomate': 'Mineuse de la tomate',

    # MALADIES FONGIQUES
    'oidium': 'Oïdium',
    'mildiou': 'Mildiou',
    'pourriture_grise': 'Pourriture grise',
    'pourriture_blanche': 'Sclérotinia',
    'rouille': 'Rouille',
}

# Appliquer le mapping
df_inference["pathologie"] = df_inference["image_key"].map(image_to_pathology)

print("\n Résultats du mapping :")
print(f"{'Image':<25s} {'Pathologie':<30s} {'Type (inférence)':<15s} {'Trouvé ?':<10s}")
print("-" * 85)

for idx, row in df_inference.iterrows():
    img = row['Image']
    patho = row['pathologie']
    type_inf = row['Type']

    # Vérifier si la pathologie existe dans df_expert
    if pd.notna(patho) and patho in df_expert['name'].tolist():
        expert_row = df_expert[df_expert['name'] == patho].iloc[0]
        type_expert = expert_row['Type']
        found = '✅'

        # Vérifier la cohérence des types
        if type_inf == type_expert or (type_inf == 'Carence' and type_expert == 'Maladie'):
            consistency = '✅'
        else:
            consistency = f'⚠️ ({type_expert})'
    else:
        found = '❌'
        consistency = ''

    print(f"{img:<25s} {patho if pd.notna(patho) else 'MANQUANT':<30s} {type_inf:<15s} {found:<10s} {consistency}")

# Identifier les images sans correspondance
missing = df_inference[df_inference['pathologie'].isna()]
if len(missing) > 0:
    print(f"\n⚠️ {len(missing)} images SANS correspondance :")
    for idx, row in missing.iterrows():
        print(f"   - {row['Image']} (Type: {row['Type']})")
else:
    print(f"\n✅ Toutes les images ({len(df_inference)}) ont une correspondance valide")

# Vérifier la présence dans df_expert
missing_in_expert = []
for img, patho in image_to_pathology.items():
    if patho not in df_expert['name'].tolist():
        missing_in_expert.append((img, patho))

if missing_in_expert:
    print(f"\n⚠️ {len(missing_in_expert)} pathologies ABSENTES de df_expert :")
    for img, patho in missing_in_expert:
        print(f"   - '{patho}' (image: {img})")
    print("\n💡 Ces pathologies doivent être ajoutées à df_expert")
else:
    print(f"\n✅ Toutes les pathologies mappées existent dans df_expert")


# VÉRIFICATION DE LA COHÉRENCE DES TYPES (VERSION ROBUSTE)

print("\n ------ VÉRIFICATION DE LA COHÉRENCE DES TYPES ----------")

# Fonction de nettoyage des types
def clean_type(type_str):
    """Nettoie et normalise les types de pathologies"""
    if pd.isna(type_str):
        return None

    # Convertir en string, supprimer espaces, lowercase
    cleaned = str(type_str).strip().lower()

    # Supprimer espaces multiples
    cleaned = ' '.join(cleaned.split())

    # Normaliser les accents (é → e)
    cleaned = unicodedata.normalize('NFD', cleaned)
    cleaned = ''.join(char for char in cleaned if unicodedata.category(char) != 'Mn')

    return cleaned

# Créer une vue comparative
comparison = df_inference.merge(
    df_expert[['name', 'Type']],
    left_on='pathologie',
    right_on='name',
    how='left',
    suffixes=('_inference', '_expert')
)

# Appliquer le nettoyage
comparison['Type_inference_clean'] = comparison['Type_inference'].apply(clean_type)
comparison['Type_expert_clean'] = comparison['Type_expert'].apply(clean_type)

# Affichage du tableau
print(f"\n{'Image':<25s} {'Type (inférence)':<20s} {'Type (expert)':<20s} {'Cohérent ?':<15s}")
print("-" * 85)

conflicts = 0
conflict_details = []

for idx, row in comparison.iterrows():
    img = row['Image']
    type_inf = row['Type_inference']
    type_exp = row['Type_expert'] if pd.notna(row['Type_expert']) else 'NON TROUVÉ'

    # Comparaison sur versions nettoyées
    type_inf_clean = row['Type_inference_clean']
    type_exp_clean = row['Type_expert_clean']

    if pd.isna(type_exp_clean):
        coherent = '❌ Manquant'
        conflicts += 1
        conflict_details.append({
            'image': img,
            'issue': 'Pathologie absente de df_expert'
        })
    elif type_inf_clean == type_exp_clean:
        coherent = '✅ Exact'
    else:
        coherent = f'❌ Conflit'
        conflicts += 1
        conflict_details.append({
            'image': img,
            'type_inf': type_inf,
            'type_exp': type_exp,
            'type_inf_clean': type_inf_clean,
            'type_exp_clean': type_exp_clean
        })

    print(f"{img:<25s} {type_inf:<20s} {type_exp:<20s} {coherent:<15s}")

# Résumé
print("\n" + "="*85)
if conflicts == 0:
    print(f"✅ PARFAIT ! Tous les types ({len(comparison)}) sont cohérents")
else:
    print(f"⚠️ {conflicts} conflits détectés sur {len(comparison)} images")

    # Afficher les détails des conflits
    if conflict_details:
        print("\n🔍 Détail des conflits :")
        for detail in conflict_details:
            if 'issue' in detail:
                print(f"\n   ❌ {detail['image']} : {detail['issue']}")
            else:
                print(f"\n   ❌ {detail['image']} :")
                print(f"      Inference : '{detail['type_inf']}' → '{detail['type_inf_clean']}'")
                print(f"      Expert    : '{detail['type_exp']}' → '{detail['type_exp_clean']}'")
                print(f"      Différence : Les types normalisés ne correspondent pas")

In [ ]:
# JOINTURE
df_merged = df_inference.merge(
    df_expert[['name', 'Type', 'reference_fr']],
    left_on='pathologie', # table inference
    right_on='name',      # table expert
    how='left',
    suffixes=('_inference', '_expert')

)
df_merged


In [ ]:
# Sélectionner les colonnes à garder
print("\n Sélection les colonnes essentielles...")

df_clean = df_merged[[
    'Image',           # Nom de l'image
    'pathologie',      # Nom de la pathologie
    'Type_expert',     # Type de la pathologie
    'Moondream',       # Sortie Moondream (FR)
    'GPT-5',          # Sortie GPT-5 (FR)
    'Moondream annotation',  # Annotation Moondream
    'GPT-5 annotation',      # Annotation GPT-5
    'reference_fr'     # Référence experte (FR)
]].copy()

# 3. Renommer 'Type_expert' en 'Type' (plus simple)
df_clean = df_clean.rename(columns={
    'Type_expert': 'Type',
    'GPT-5': 'GPT5', # Optionnel: enlever le tiret
    'GPT-5 annotation': 'GPT5 annotation', # Optionnel: enlever le tiret
})

df_clean

In [ ]:
# Sauvegarder le dataset nettoyé
df_clean.to_csv('/content/drive/MyDrive/PROJET_DJMA/djma_inference.csv', index=False, encoding='utf-8')
print(f"Dataset sauvegardé : /content/drive/MyDrive/PROJET_DJMA/djma_inference.csv")

# Phase 2 - Prétraitement linguistique

traduction Fr --> En

analyse longueur : choix du max_length

## TRADUCTION

In [ ]:
print(f"Nombre de textes à traduire :")
print(f"   - Références expertes : {len(df_clean)} textes")
print(f"   - Moondream : {(df_clean['Moondream'].notna()).sum()} textes")
print(f"   - GPT5 : {(df_clean['GPT5'].notna()).sum()} textes")
print(f"\n   Total : {len(df_clean) * 3} traductions à effectuer")

total_translations = len(df_clean) * 3

In [ ]:
print("\n ------ TRADUCTION FR → EN -------")

# Initialiser le traducteur
translator = GoogleTranslator(source='fr', target='en')

def translate_with_retry(text, max_retries=3, delay=1):
    """
    Traduit un texte du français vers l'anglais avec gestion d'erreurs.

    Args:
        text: Texte à traduire
        max_retries: Nombre de tentatives en cas d'échec
        delay: Délai entre les tentatives (secondes)

    Returns:
        str: Texte traduit ou None en cas d'échec
    """
    if pd.isna(text) or text.strip() == "":
        return ""

    for attempt in range(max_retries):
        try:
            result = translator.translate(text)
            time.sleep(0.3)  # Pause pour éviter rate limiting
            return result

        except Exception as e:
          if attempt < max_retries - 1:
            print(f"⚠️ Erreur (tentative {attempt+1}/{max_retries}): {str(e)[:50]}...")
            time.sleep(delay * (attempt + 1))
          else:
            print("❌ Échec définitif")
            return None

# TRADUCTION

print("\n 1/3 Traduction des références expertes...")
print("-" * 60)

df_clean['reference_en'] = None  # Initialiser la colonne

for idx, row in df_clean.iterrows():
    print(f"   [{idx+1}/{len(df_clean)}] {row['pathologie'][:30]:<30s}", end=" ")

    translated = translate_with_retry(row['reference_fr'])
    df_clean.at[idx, 'reference_en'] = translated

    if translated:
        print("✅")
    else:
        print("❌ ÉCHEC")

print("\n✅ Références expertes traduites")

# ---

print("\n 2/3 Traduction Moondream...")
print("-" * 60)

df_clean['moondream_en'] = None

for idx, row in df_clean.iterrows():
    print(f"   [{idx+1}/{len(df_clean)}] {row['Image'][:30]:<30s}", end=" ")

    translated = translate_with_retry(row['Moondream'])
    df_clean.at[idx, 'moondream_en'] = translated

    if translated:
        print("✅")
    else:
        print("❌ ÉCHEC")

print("\n✅ Moondream traduit")

# ---

print("\n 3/3 Traduction GPT5...")
print("-" * 60)

df_clean['gpt5_en'] = None

for idx, row in df_clean.iterrows():
    print(f"   [{idx+1}/{len(df_clean)}] {row['Image'][:30]:<30s}", end=" ")

    translated = translate_with_retry(row['GPT5'])
    df_clean.at[idx, 'gpt5_en'] = translated

    if translated:
        print("✅")
    else:
        print("❌ ÉCHEC")

print("\n GPT5 traduit")

print("\n ------ TRADUCTION TERMINÉE -------")

# Vérifier les échecs
failed_refs = df_clean['reference_en'].isna().sum()
failed_moondream = df_clean['moondream_en'].isna().sum()
failed_gpt5 = df_clean['gpt5_en'].isna().sum()

print(f"\n Résultats :")
print(f"   - Références : {len(df_clean) - failed_refs}/{len(df_clean)} réussies")
print(f"   - Moondream  : {len(df_clean) - failed_moondream}/{len(df_clean)} réussies")
print(f"   - GPT5       : {len(df_clean) - failed_gpt5}/{len(df_clean)} réussies")

if failed_refs + failed_moondream + failed_gpt5 == 0:
    print("\n✅ Toutes les traductions ont réussi !")
else:
    print(f"\n⚠️ {failed_refs + failed_moondream + failed_gpt5} traductions ont échoué")
    print(" Vous pouvez les traduire manuellement ou réessayer")

In [ ]:
print("\n ---- VÉRIFICATION QUALITATIVE DES TRADUCTIONS ------- ")
# Afficher 2 exemples complets
for i in range(min(2, len(df_clean))):
    print(f"\n{'='*60}")
    print(f"EXEMPLE {i+1} : {df_clean.iloc[i]['pathologie']} ({df_clean.iloc[i]['Type']})")
    print(f"{'='*60}")

    print(f"\n RÉFÉRENCE EXPERTE :")
    print(f"   FR : {df_clean.iloc[i]['reference_fr']}")
    print(f"   EN : {df_clean.iloc[i]['reference_en']}")

    print(f"\n MOONDREAM :")
    print(f"   FR : {df_clean.iloc[i]['Moondream']}")
    print(f"   EN : {df_clean.iloc[i]['moondream_en']}")

    print(f"\n GPT5 :")
    print(f"   FR : {df_clean.iloc[i]['GPT5']}")
    print(f"   EN : {df_clean.iloc[i]['gpt5_en']}")

In [ ]:
# DICTIONNAIRE DE CORRECTIONS POST-TRADUCTION
# Basé sur la terminologie officielle en phytopathologie

PHYTO_CORRECTIONS = {

    # Cul noir : traduction littérale inacceptable
    "Black ass":                    "Blossom end rot",
    "black ass":                    "blossom end rot",

    # Aleurodes : "midges" = moucherons, pas aleurodes
    "white midges":                 "whiteflies",
    "White midges":                 "Whiteflies",

    # Pucerons : "moults" imprécis, "cast skins" est le terme usuel
    "white moults":                 "white cast skins",

    # "sheet" = feuille de papier, pas feuille de plante
    "The sheet has":                "The leaf has",
    "the sheet has":                "the leaf has",

    # "ribs" = côtes (cuisine), "veins" = nervures végétales
    "along the ribs":               "along the veins",

    # "upper" traduit "tige" par erreur
    "The upper also shows":         "The stem also shows",
    "the upper also shows":         "the stem also shows",

    # Mildiou : oomycète, pas un vrai champignon
    "parasitic fungi.\nSmall":      "oomycetes (water molds). Small",

    # Sclérotinia : "crown rot" = pourriture du collet ≠ Sclérotinia
    "Sclerotinia or crown rot":     "Sclerotinia or white mold (stem rot)",
}

def post_correct_translation(text: str, corrections: dict = PHYTO_CORRECTIONS) -> str:
    """
    Applique les corrections post-traduction sur un texte anglais.
    Les corrections sont appliquées dans l'ordre du dictionnaire.
    """
    if pd.isna(text):
        return text
    for wrong, right in corrections.items():
        text = text.replace(wrong, right)
    return text


# APPLICATION
cols_to_correct = ['reference_en', 'moondream_en', 'gpt5_en']
print(" Application des corrections phytopathologiques...\n")

for col in cols_to_correct:
    original = df_clean[col].copy()
    df_clean[col] = df_clean[col].apply(post_correct_translation)

    # Compter les corrections appliquées
    changes = (original != df_clean[col]).sum()
    print(f"   {col:<20s} : {changes} correction(s) appliquée(s)")

# VÉRIFICATION CIBLÉE
print("\n🔍 Vérification des cas critiques :\n")

checks = {
    "Cul noir - reference_en":  ("Black ass",       df_clean.loc[df_clean['Image']=='Cul noir', 'reference_en'].values[0]),
    "Cul noir - moondream_en":  ("The upper",        df_clean.loc[df_clean['Image']=='Cul noir', 'moondream_en'].values[0]),
    "Aleurodes - reference_en": ("white midges",     df_clean.loc[df_clean['Image']=='Aleurodes', 'reference_en'].values[0]),
    "Puceron - moondream_en":   ("along the ribs",   df_clean.loc[df_clean['Image']=='Puceron', 'moondream_en'].values[0]),
}

for label, (bad_term, text) in checks.items():
    status = "✅ Corrigé" if bad_term not in text else f"⚠️ '{bad_term}' encore présent"
    print(f"   {label:<30s} → {status}")

# Sauvegarder
df_clean.to_csv('/content/drive/MyDrive/PROJET_DJMA/djma_inference_translated.csv', index=False, encoding='utf-8')
print(f"\n Dataset sauvegardé : /content/drive/MyDrive/PROJET_DJMA/djma_inference_translated.csv")


In [ ]:
print("\n ---- VÉRIFICATION QUALITATIVE DES TRADUCTIONS ------- ")
# Afficher 2 exemples complets
for i in range(min(2, len(df_clean))):
    print(f"\n{'='*60}")
    print(f"EXEMPLE {i+1} : {df_clean.iloc[i]['pathologie']} ({df_clean.iloc[i]['Type']})")
    print(f"{'='*60}")

    print(f"\n RÉFÉRENCE EXPERTE :")
    print(f"   FR : {df_clean.iloc[i]['reference_fr']}")
    print(f"   EN : {df_clean.iloc[i]['reference_en']}")

    print(f"\n MOONDREAM :")
    print(f"   FR : {df_clean.iloc[i]['Moondream']}")
    print(f"   EN : {df_clean.iloc[i]['moondream_en']}")

    print(f"\n GPT5 :")
    print(f"   FR : {df_clean.iloc[i]['GPT5']}")
    print(f"   EN : {df_clean.iloc[i]['gpt5_en']}")

# Vérifier que les traductions ne sont pas vides
print("\n ------- STATISTIQUES DES TRADUCTIONS -------")

print("\n  Longueur moyenne (caractères) :")
print(f"   Références FR : {df_clean['reference_fr'].str.len().mean():.0f}")
print(f"   Références EN : {df_clean['reference_en'].str.len().mean():.0f}")
print(f"   Moondream FR  : {df_clean['Moondream'].str.len().mean():.0f}")
print(f"   Moondream EN  : {df_clean['moondream_en'].str.len().mean():.0f}")
print(f"   GPT5 FR       : {df_clean['GPT5'].str.len().mean():.0f}")
print(f"   GPT5 EN       : {df_clean['gpt5_en'].str.len().mean():.0f}")

# Ratio FR/EN (doit être proche de 1.0-1.2)
ref_ratio = df_clean['reference_en'].str.len().mean() / df_clean['reference_fr'].str.len().mean()
print(f"\n Ratio longueur EN/FR :")
print(f"   Références : {ref_ratio:.2f} (attendu: 0.9-1.1)")

if 0.8 <= ref_ratio <= 1.2:
    print("✅ Ratio normal")
else:
    print("⚠️ Ratio inhabituel, vérifier les traductions")

##  ANALYSE DES LONGUEURS (tokens)

In [ ]:
print(" ---- ANALYSE DES LONGUEURS (tokens DistilBERT) ----- ")
# Charger le tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
def count_tokens(text):
    """Compte le nombre de tokens (avec tokens spéciaux [CLS], [SEP])"""
    if pd.isna(text) or text.strip() == "":
        return 0
    return len(tokenizer.encode(text, add_special_tokens=True))

def count_pair_tokens(premise, hypothesis):
    """
    Compte les tokens d'une paire NLI concaténée :
    [CLS] premise [SEP] hypothesis [SEP]
    C'est la vraie longueur vue par le modèle NLI.
    """
    if pd.isna(premise) or pd.isna(hypothesis):
        return 0
    encoding = tokenizer(
        premise, hypothesis,
        truncation=False,
        add_special_tokens=True
    )
    return len(encoding['input_ids'])

# TOKENS INDIVIDUELS
print("\n Calcul des tokens individuels...")

df_clean['ref_tokens']       = df_clean['reference_en'].apply(count_tokens)
df_clean['moondream_tokens'] = df_clean['moondream_en'].apply(count_tokens)
df_clean['gpt5_tokens']      = df_clean['gpt5_en'].apply(count_tokens)

print("\n Statistiques de longueur (tokens individuels) :")
print("\n", df_clean[['ref_tokens', 'moondream_tokens', 'gpt5_tokens']].describe())

# TOKENS DES PAIRES NLI
print("\n Calcul des tokens des paires NLI (premise + hypothesis)...")

df_clean['pair_moondream_tokens'] = df_clean.apply(
    lambda row: count_pair_tokens(row['reference_en'], row['moondream_en']), axis=1
)
df_clean['pair_gpt5_tokens'] = df_clean.apply(
    lambda row: count_pair_tokens(row['reference_en'], row['gpt5_en']), axis=1
)

print("\n Statistiques de longueur (tokens paires NLI) :")
print("\n", df_clean[['pair_moondream_tokens', 'pair_gpt5_tokens']].describe())

# VÉRIFICATION LIMITE 128
MAX_LEN = 128

print(f"\n⚠️ Séquences dépassant {MAX_LEN} tokens :")
print(f"\n   [INDIVIDUELS]")
print(f"   Références : {(df_clean['ref_tokens'] > MAX_LEN).sum()} / {len(df_clean)}")
print(f"   Moondream  : {(df_clean['moondream_tokens'] > MAX_LEN).sum()} / {len(df_clean)}")
print(f"   GPT5       : {(df_clean['gpt5_tokens'] > MAX_LEN).sum()} / {len(df_clean)}")

print(f"\n   [PAIRES NLI - vue réelle du modèle]")
moon_over = (df_clean['pair_moondream_tokens'] > MAX_LEN).sum()
gpt5_over = (df_clean['pair_gpt5_tokens'] > MAX_LEN).sum()
print(f"   Paires ref+Moondream : {moon_over} / {len(df_clean)}")
print(f"   Paires ref+GPT5      : {gpt5_over} / {len(df_clean)}")

# Détail des paires longues
long_pairs = df_clean[
    (df_clean['pair_moondream_tokens'] > MAX_LEN) |
    (df_clean['pair_gpt5_tokens'] > MAX_LEN)
]

if len(long_pairs) > 0:
    print(f"\n   Détail des {len(long_pairs)} paires à risque de troncature :")
    print(long_pairs[[
        'Image', 'pathologie',
        'ref_tokens', 'moondream_tokens', 'gpt5_tokens',
        'pair_moondream_tokens', 'pair_gpt5_tokens'
    ]].to_string(index=False))
else:
    print(f"\n✅ Toutes les paires NLI tiennent dans {MAX_LEN} tokens")

# RECOMMANDATION max_length
# Basée sur les paires (longueur réelle vue par le modèle)
all_pair_tokens = pd.concat([
    df_clean['pair_moondream_tokens'],
    df_clean['pair_gpt5_tokens']
])

max_95  = int(all_pair_tokens.quantile(0.95))
max_100 = int(all_pair_tokens.max())

print(f"\n  RECOMMANDATIONS max_length (basées sur les paires NLI) :")
print(f"   - Pour couvrir 95% des paires  : {max_95} tokens")
print(f"   - Pour couvrir 100% des paires : {max_100} tokens")

if max_100 <= 128:
    RECOMMENDED_MAX_LENGTH = 128
    print(f"\n✅ Recommandation : max_length = {RECOMMENDED_MAX_LENGTH} (compatible SNLI)")
elif max_100 <= 256:
    RECOMMENDED_MAX_LENGTH = 256
    print(f"\n⚠️ Recommandation : max_length = {RECOMMENDED_MAX_LENGTH} (dépasse SNLI, vérifier le modèle)")
else:
    RECOMMENDED_MAX_LENGTH = 512
    print(f"\n⚠️ Recommandation : max_length = {RECOMMENDED_MAX_LENGTH} (séquences très longues)")

print(f"\n   → RECOMMENDED_MAX_LENGTH = {RECOMMENDED_MAX_LENGTH}")

In [ ]:
print(" ---- Création de la visualisation des paires NLI -------")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Couleurs
COLOR_MOON = '#2ecc71'
COLOR_GPT5 = '#e74c3c'
MAX_LEN = 128

# Graphique 1 : Paires ref + Moondream
axes[0].hist(df_clean['pair_moondream_tokens'], bins=10,
             color=COLOR_MOON, alpha=0.7, edgecolor='black')
axes[0].axvline(x=MAX_LEN, color='red', linestyle='--',
                linewidth=2, label=f'Limite SNLI ({MAX_LEN})')

# Annotation : nb de paires au-dessus
n_over = (df_clean['pair_moondream_tokens'] > MAX_LEN).sum()
axes[0].annotate(f'{n_over}/{len(df_clean)} paires\ndépassent {MAX_LEN}',
                 xy=(MAX_LEN, axes[0].get_ylim()[1] if False else 1),
                 xytext=(MAX_LEN + 5, 2.5),
                 color='red', fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='red'))

axes[0].set_title('Paires NLI : Référence + Moondream',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Nombre de tokens (paire complète)', fontsize=11)
axes[0].set_ylabel('Fréquence', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Stats dans le graphique
mean_m = df_clean['pair_moondream_tokens'].mean()
axes[0].axvline(x=mean_m, color='#27ae60', linestyle=':',
                linewidth=1.5, label=f'Moyenne ({mean_m:.0f})')
axes[0].legend(fontsize=10)

# Graphique 2 : Paires ref + GPT5
axes[1].hist(df_clean['pair_gpt5_tokens'], bins=10,
             color=COLOR_GPT5, alpha=0.7, edgecolor='black')
axes[1].axvline(x=MAX_LEN, color='red', linestyle='--',
                linewidth=2, label=f'Limite SNLI ({MAX_LEN})')

n_over2 = (df_clean['pair_gpt5_tokens'] > MAX_LEN).sum()
axes[1].annotate(f'{n_over2}/{len(df_clean)} paires\ndépassent {MAX_LEN}',
                 xy=(MAX_LEN, 1),
                 xytext=(MAX_LEN + 5, 2.5),
                 color='red', fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='red'))

axes[1].set_title('Paires NLI : Référence + GPT-5',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nombre de tokens (paire complète)', fontsize=11)
axes[1].set_ylabel('Fréquence', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

mean_g = df_clean['pair_gpt5_tokens'].mean()
axes[1].axvline(x=mean_g, color='#c0392b', linestyle=':',
                linewidth=1.5, label=f'Moyenne ({mean_g:.0f})')
axes[1].legend(fontsize=10)

# Titre global
plt.suptitle(
    'Distribution des longueurs (tokens) — Paires NLI complètes\n'
    '[CLS] référence [SEP] hypothèse [SEP]',
    fontsize=14, fontweight='bold'
)

plt.tight_layout()
#plt.savefig('/content/drive/MyDrive/PROJET_DJMA/pair_token_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
#print("✅ Graphique sauvegardé")

In [ ]:
# Colonnes actuelles
print(f"\n Colonnes actuelles ({len(df_clean.columns)}) :")
print(df_clean.columns.tolist())

# Sélectionner uniquement les colonnes nécessaires pour NLI
df_final = df_clean[[
    'Image',           # Identifiant de l'image
    'pathologie',      # Nom de la pathologie
    'Type',           # Type (Maladie/Ravageur/Carence)
    'reference_en',    # Référence experte (EN) - PREMISE
    'moondream_en',    # Moondream (EN) - HYPOTHESIS 1
    'gpt5_en',         # GPT5 (EN) - HYPOTHESIS 2
    'Moondream annotation', # Annotation Moondream
    'GPT5 annotation'       # Annotation GPT5
]].copy()

print(f"\n✅ Colonnes sélectionnées ({len(df_final.columns)}) :")
print(df_final.columns.tolist())

# Vérifier qu'il n'y a pas de valeurs manquantes
print("\n Valeurs manquantes :")
missing = df_final.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ Aucune valeur manquante")


# Exemple détaillé
print("\n--- Exemple détaillé (première ligne) ---")
print(df_final.iloc[0])

# Sauvegarder le dataset final
df_final.to_csv('/content/drive/MyDrive/PROJET_DJMA/djma_final_for_nli.csv', index=False, encoding='utf-8')
print(f"\n Dataset final sauvegardé : /content/drive/MyDrive/PROJET_DJMA/djma_final_for_nli.csv")

print("\n ------ DATASET PRÊT POUR L'INFÉRENCE NLI ------ ")

print(f"""
RÉSUMÉ FINAL :
   - Images : {len(df_final)}
   - Colonnes : {len(df_final.columns)}
   - Pathologies : {df_final['pathologie'].nunique()}
   - Types : {', '.join(df_final['Type'].unique())}

Structure des paires NLI :
   - PREMISE (référence) : reference_en
   - HYPOTHESIS (modèle)  : moondream_en OU gpt5_en

Prêt pour :
   1. Construction des paires NLI
   2. Inférence avec modèle DistilBERT+LoRA
   3. Analyse comparative Moondream vs GPT5
""")

# Phase 3 - Inférence NLI
Construction des paires NLI

Chargement du modele NLI

Fonction d'inférence NLI

Inférence sur les paires

Analyse des résultats

 ## Construction des paires NLI

In [ ]:
# Créer les paires (premise, hypothesis) pour chaque modèle
pairs = []

for idx, row in df_final.iterrows():
    premise = row['reference_en']  # Référence experte = PREMISE

    # Paire 1 : Moondream
    pairs.append({
        'id': f"{idx}_moondream",
        'image': row['Image'],
        'pathologie': row['pathologie'],
        'type': row['Type'],
        'premise': premise,
        'hypothesis': row['moondream_en'],
        'annotation': row['Moondream annotation'],
        'model_source': 'Moondream'
    })

    # Paire 2 : GPT-5
    pairs.append({
        'id': f"{idx}_gpt5",
        'image': row['Image'],
        'pathologie': row['pathologie'],
        'type': row['Type'],
        'premise': premise,
        'hypothesis': row['gpt5_en'],
        'annotation': row['GPT5 annotation'],
        'model_source': 'GPT-5'
    })

df_pairs = pd.DataFrame(pairs)

print(f"\n✅ {len(df_pairs)} paires NLI créées")
print(f"\nRépartition par modèle :")
print(df_pairs['model_source'].value_counts())

print(f"\nRépartition par type de pathologie :")
print(df_pairs['type'].value_counts())

# Aperçu des paires
print("\n--- Aperçu des paires ---")
print(df_pairs[['image', 'pathologie', 'model_source']].head(6))

# Exemple détaillé d'une paire
print("\n--- Exemple de paire NLI ---")
example = df_pairs.iloc[0]
print(f"\nImage      : {example['image']}")
print(f"Pathologie : {example['pathologie']}")
print(f"Modèle     : {example['model_source']}")
print(f"Type       : {example['type']}")
print(f"Annotation : {example['annotation']}")
print(f"\n PREMISE (référence experte) :")
print(f"   {example['premise']}")
print(f"\n HYPOTHESIS ({example['model_source']}) :")
print(f"   {example['hypothesis']}")

# Sauvegarder les paires
#df_pairs.to_csv('/content/drive/MyDrive/PROJET_DJMA/nli_pairs.csv', index=False, encoding='utf-8')
#print(f" Paires sauvegardées : /content/drive/MyDrive/PROJET_DJMA/nli_pairs.csv")

## Chargement du modèle NLI

In [ ]:
# Charger le meilleur checkpoint (EPOCH 10 - F1 = 89.04%)
best_checkpoint = "/content/drive/MyDrive/PROJET_DJMA/models/distilbert_lora_snli_nli/checkpoint-171680"
print(f" Chargement du checkpoint : {best_checkpoint}")

# Charger le modèle de base
model_base = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

# Charger les adaptateurs LoRA
model = PeftModel.from_pretrained(model_base, best_checkpoint)
model.eval()
model.to(device)

print(f" Modèle chargé et prêt pour l'inférence")
print(f"   Device : {model.device}")


## Fonction d'inférence NLI


In [ ]:
# FONCTION show_truncated_input (avec stratégie)
def show_truncated_input(premise, hypothesis, tokenizer,
                          max_length=128, strategy='only_first'):
    """
    Affiche le texte réellement vu par le modèle après troncature
    selon la stratégie choisie.

    Args:
        premise    : Texte de référence experte (EN)
        hypothesis : Texte généré par le modèle (EN)
        tokenizer  : Tokenizer DistilBERT
        max_length : Longueur maximale (128 = limite SNLI)
        strategy   : 'only_first' | 'longest_first' | 'only_second'
    """
    tokens_premise    = count_tokens(premise)
    tokens_hypothesis = count_tokens(hypothesis)
    tokens_pair       = count_pair_tokens(premise, hypothesis)
    tokens_lost       = max(0, tokens_pair - max_length)

    encoding = tokenizer(
        premise, hypothesis,
        truncation=strategy,
        max_length=max_length,
        add_special_tokens=True
    )

    decoded = tokenizer.decode(encoding['input_ids'], skip_special_tokens=False)
    parts   = decoded.split('[SEP]')

    premise_seen    = parts[0].replace('[CLS]', '').strip()
    hypothesis_seen = parts[1].strip() if len(parts) > 1 else 'N/A'

    tokens_premise_seen    = len(tokenizer.encode(premise_seen,    add_special_tokens=False))
    tokens_hypothesis_seen = len(tokenizer.encode(hypothesis_seen, add_special_tokens=False))

    print(f"   {'─'*55}")
    print(f"   Stratégie        : {strategy}")
    print(f"   Tokens utilisés  : {len(encoding['input_ids'])} / {max_length} "
          f"({tokens_lost} perdus)")
    print(f"   Premise          : {tokens_premise} → {tokens_premise_seen} tokens "
          f"({'tronquée ⚠️' if tokens_premise_seen < tokens_premise else 'intacte ✅'})")
    print(f"   Hypothesis       : {tokens_hypothesis} → {tokens_hypothesis_seen} tokens "
          f"({'tronquée ⚠️' if tokens_hypothesis_seen < tokens_hypothesis else 'intacte ✅'})")
    print(f"\n   PREMISE vue par le modèle :")
    print(f"   → {premise_seen}")
    print(f"\n   HYPOTHESIS vue par le modèle :")
    print(f"   → {hypothesis_seen}")

# TOP 3 PAIRES CRITIQUES — Toutes stratégies

STRATEGIES = ['only_first', 'longest_first', 'only_second']

def get_critical_cases(df_clean, tokenizer, top_n=3):
    """
    Identifie les N paires les plus tronquées du dataset.

    Args:
        df_clean  : DataFrame nettoyé
        tokenizer : Tokenizer DistilBERT
        top_n     : Nombre de cas critiques à retourner

    Returns:
        DataFrame : top_n lignes triées par perte maximale
    """
    return (
        df_clean[['Image', 'reference_en', 'moondream_en', 'gpt5_en']]
        .assign(
            loss_moon=lambda d: d.apply(
                lambda r: max(0, count_pair_tokens(r['reference_en'], r['moondream_en']) - 128),
                axis=1),
            loss_gpt5=lambda d: d.apply(
                lambda r: max(0, count_pair_tokens(r['reference_en'], r['gpt5_en']) - 128),
                axis=1)
        )
        .assign(max_loss=lambda d: d[['loss_moon', 'loss_gpt5']].max(axis=1))
        .nlargest(top_n, 'max_loss')
    )


def analyze_critical_cases(critical_cases, model, tokenizer,
                             device, max_length=128):
    """
    Pour chaque cas critique, compare les 3 stratégies de troncature
    sur Moondream et GPT5.

    Args:
        critical_cases : DataFrame retourné par get_critical_cases()
        model          : Modèle NLI
        tokenizer      : Tokenizer DistilBERT
        device         : 'cuda' ou 'cpu'
        max_length     : Longueur maximale
    """
    for _, row in critical_cases.iterrows():
        print(f"\n ------ {row['Image']} — perte max = {row['max_loss']} tokens ------")


        for model_col, model_label in [('moondream_en', 'Moondream'),
                                        ('gpt5_en',      'GPT5')]:
            loss = max(0, count_pair_tokens(row['reference_en'], row[model_col]) - 128)

            if loss == 0:
                print(f"\n  [{model_label}] → pas de troncature ✅")
                continue

            print(f"\n  [{model_label}] — {loss} tokens perdus")

            for strategy in STRATEGIES:
                show_truncated_input(
                    row['reference_en'], row[model_col],
                    tokenizer, max_length=max_length, strategy=strategy
                )
                print()

# Exécution
print("\n Identification des cas critiques...")
critical_cases = get_critical_cases(df_clean, tokenizer, top_n=3)

print(f"\n   Top 3 paires les plus tronquées :")
print(critical_cases[['Image', 'loss_moon', 'loss_gpt5', 'max_loss']].to_string(index=False))

print(f"\n -------- ANALYSE DÉTAILLÉE — Top 3 cas critiques --------")

analyze_critical_cases(
    critical_cases,
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=128
)


- only_first

Avantage : l'hypothesis (sortie modèle à évaluer) est toujours préservée.
Risque : la référence experte peut être trop amputée pour être utile.

-  only_second

Avantage : la référence experte est toujours préservée intégralement.
Risque : la sortie du modèle à évaluer est coupée → NLI peut manquer le diagnostic clé.

- longest_only

Avantage : perte équilibrée entre les deux textes.
Risque : les deux sont tronqués → comportement le moins prévisible.

---

### Décision :
only_first reste le meilleur choix pour le pipeline car le NLI doit juger si l'hypothesis (sortie Moondream/GPT5) est consistante avec la prémise (référence experte). Il faut donc que l'hypothesis soit vue en entier par le modèle, only_first fait exactement ça.

In [ ]:
# FONCTION predict_nli
def predict_nli(
    premise: str,
    hypothesis: str,
    model,
    tokenizer,
    max_length: int = 128,
    device: str = 'cuda',
    truncation_strategy: str = 'only_first',
    verbose_truncation: bool = False
) -> dict:
    """
    Prédit la relation NLI entre premise et hypothesis.

    Args:
        premise              : Texte de référence experte (EN)
        hypothesis           : Texte généré par le modèle (EN)
        model                : Modèle DistilBERT+LoRA chargé
        tokenizer            : Tokenizer DistilBERT
        max_length           : Longueur maximale (128 = limite SNLI)
        device               : 'cuda' ou 'cpu'
        truncation_strategy  : 'only_first' (défaut) | 'longest_first' | 'only_second'
        verbose_truncation   : Affiche un warning si troncature appliquée

    Returns:
        dict: {
            'predicted_label'    : str,   # 'entailment' | 'neutral' | 'contradiction'
            'prob_entailment'    : float,
            'prob_neutral'       : float,
            'prob_contradiction' : float,
            'confidence'         : float, # prob de la classe prédite
            'confidence_gap'     : float, # écart top-1 vs top-2
            'tokens_used'        : int,   # tokens réellement encodés
            'tokens_truncated'   : int,   # tokens perdus (0 si pas de troncature)
            'truncated'          : bool   # True si troncature appliquée
        }
    """
    # ── Calcul pré-troncature pour logging ────────────────
    full_encoding    = tokenizer(premise, hypothesis,
                                 truncation=False, add_special_tokens=True)
    full_length      = len(full_encoding['input_ids'])
    tokens_truncated = max(0, full_length - max_length)
    truncated        = tokens_truncated > 0

    if verbose_truncation and truncated:
        pct = round(tokens_truncated / full_length * 100, 1)
        print(f"   ⚠️ Troncature : {full_length} → {max_length} tokens "
              f"({tokens_truncated} perdus, {pct}% de la paire)")

    # Tokenisation avec troncature choisie
    inputs = tokenizer(
        premise, hypothesis,
        return_tensors="pt",
        truncation=truncation_strategy,
        max_length=max_length,
        padding=True
    )
    tokens_used = inputs['input_ids'].shape[1]

    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Inférence
    with torch.no_grad():
        outputs       = model(**inputs)
        logits        = outputs.logits
        probs         = torch.softmax(logits, dim=-1)[0]
        predicted_idx = logits.argmax(-1).item()

    # Confidence gap (top-1 vs top-2)
    probs_sorted   = torch.sort(probs, descending=True)[0]
    confidence_gap = (probs_sorted[0] - probs_sorted[1]).item()

    labels_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

    return {
        'predicted_label'    : labels_map[predicted_idx],
        'prob_entailment'    : probs[0].item(),
        'prob_neutral'       : probs[1].item(),
        'prob_contradiction' : probs[2].item(),
        'confidence'         : probs[predicted_idx].item(),
        'confidence_gap'     : confidence_gap,
        'tokens_used'        : tokens_used,
        'tokens_truncated'   : tokens_truncated,
        'truncated'          : truncated,
    }

print("✅ predict_nli définie")

In [ ]:
print("\n--- Test predict_nli (cas standard) ---")

test_result = predict_nli(
    premise    = "A cat is on a table",
    hypothesis = "An animal is indoors",
    model      = model,
    tokenizer  = tokenizer,
    device     = device,
    verbose_truncation = True
)

print("\n Résultat :")
for key, value in test_result.items():
    if isinstance(value, float):
        print(f"   {key:<22s} : {value:.4f}")
    else:
        print(f"   {key:<22s} : {value}")

## Inférence sur toutes les paires

In [ ]:
print(" ---- INFÉRENCE NLI SUR LES PAIRES DJMA-MODELE ----")

results = []
print(f"\n Traitement de {len(df_pairs)} paires...")
print("-" * 10)

label_symbols = {'entailment': '✅', 'neutral': '⚠️', 'contradiction': '❌'}

for idx, row in df_pairs.iterrows():
    ann_symbol = "✅V" if row["annotation"] is True else ("❌F" if row["annotation"] is False else "❓")

    # Progress
    print(f"[{idx+1:2d}/{len(df_pairs)}] {row['pathologie'][:25]:<25s} | {row['model_source']:<10s} | ann={ann_symbol}", end=" ")

    pred = predict_nli(
        premise=row['premise'],
        hypothesis=row['hypothesis'],
        model=model,
        tokenizer=tokenizer,
        max_length=128,
        device=device
    )

    symbol = label_symbols[pred['predicted_label']]
    print(f"{symbol} {pred['predicted_label']:<15s} (conf={pred['confidence']:.3f}, gap={pred['confidence_gap']:.3f})")

    results.append({
        'id': row['id'],
        'image': row['image'],
        'pathologie': row['pathologie'],
        'type': row['type'],
        'model_source': row['model_source'],
        'predicted_label': pred['predicted_label'],
        'prob_entailment': pred['prob_entailment'],
        'prob_neutral': pred['prob_neutral'],
        'prob_contradiction': pred['prob_contradiction'],
        'confidence': pred['confidence'],
        'confidence_gap': pred['confidence_gap'],
        'annotation': row['annotation'],
        'annotation_vf': "V" if row["annotation"] is True else ("F" if row["annotation"] is False else "")
    })

df_results = pd.DataFrame(results)

print("\n ------ INFÉRENCE TERMINÉE -----")
print(f"\n {len(df_results)} prédictions effectuées")

df_results.to_csv('/content/drive/MyDrive/PROJET_DJMA/nli_results.csv', index=False, encoding='utf-8')
print(" Résultats sauvegardés : /content/drive/MyDrive/PROJET_DJMA/nli_results.csv")

##  Analyse des résultats

In [ ]:
print("\n ---- ANALYSE DES RÉSULTATS NLI ----- ")

# 1. Distribution globale des prédictions
print("\n 1. DISTRIBUTION GLOBALE DES PRÉDICTIONS")
print("-" *10)

distribution = df_results['predicted_label'].value_counts()
print(distribution)
print(f"\nPourcentages :")
print((distribution / len(df_results) * 100).round(1))

# 2. Comparaison par modèle
print("\n 2. COMPARAISON MOONDREAM vs GPT-5")
print("-" * 10)

comparison = df_results.groupby('model_source')['predicted_label'].value_counts(normalize=True).unstack(fill_value=0) * 100

print("\n Distribution des prédictions (%) :")
print(comparison.round(1))


# Aligner label NLI avec annotation humaine
# Convention : annotation True (V) → on attend entailment
#              annotation False (F) → on attend neutral ou contradiction

def nli_matches_annotation(row):
    """
    Retourne True si la prédiction NLI est cohérente avec l'annotation humaine.
    V (True)  → entailment attendu
    F (False) → neutral ou contradiction attendu
    """
    ann = row['annotation']
    label = row['predicted_label']
    if ann is True:
        return label == 'entailment'
    elif ann is False:
        return label in ('neutral', 'contradiction')
    return None  # annotation inconnue

df_results['nli_correct'] = df_results.apply(nli_matches_annotation, axis=1)

# 3. Accuracy par modèle
accuracy = df_results.groupby('model_source')['nli_correct'].mean()
print("\n ------ 3. Accuracy NLI vs annotations humaines : ------")
print(accuracy.round(3))

# Accuracy par modèle ET par type de pathologie
accuracy_by_type = df_results.groupby(['model_source', 'type'])['nli_correct'].mean().unstack()
print("\nAccuracy par type de pathologie :")
print(accuracy_by_type.round(3))

# 4. Tableau cas par cas — la pièce maîtresse
print("\n 4. Détail paire par paire :")
print("-" * 10)
detail = df_results[[
    'image', 'model_source', 'annotation',
    'predicted_label', 'confidence', 'nli_correct'
]].copy()
detail['annotation_str'] = detail['annotation'].map({True: 'V', False: 'F'})
detail['correct_str'] = detail['nli_correct'].map({True: '✅', False: '❌'})
print(detail[['image','model_source','annotation_str',
              'predicted_label','confidence','correct_str']].to_string(index=False))

# 5. Analyse des erreurs — comprendre POURQUOI le modèle se trompe
print("\n 5. Cas mal prédits :")
print("-" * 10)
errors = df_results[df_results['nli_correct'] == False]
print(errors[['image','model_source','annotation',
              'predicted_label','confidence']].to_string(index=False))

# 6. Score de qualité CORRIGÉ (pondéré par annotation)
# Récompense entailment seulement si annotation=V
# Pénalise entailment si annotation=F
def corrected_score(row):
    if row['annotation'] is True:
        return row['prob_entailment'] - row['prob_contradiction']
    else:  # annotation F → neutral/contradiction attendu
        return row['prob_neutral'] + row['prob_contradiction'] - row['prob_entailment']

df_results['corrected_score'] = df_results.apply(corrected_score, axis=1)
print("\n 6. Score corrigé par modèle :")
print("-" * 10)
print(df_results.groupby('model_source')['corrected_score'].agg(['mean','std']).round(3))

### CONCLUSION
Le score corrigé confirme sans ambiguité : GPT-5 à +0.285 contre Moondream à -0.215. Le signe négatif de Moondream signifie qu'il est activement pénalisé? il prédit entailment sur des cas annotés F plus souvent qu'il ne prédit neutral/contradiction. C'est le biais de permissivité confirmé chiffré.

L'accuracy le dit aussi : GPT-5 50% vs Moondream 33%.

Ce que le tableau cas par cas révèle.
Il y a 3 patterns distincts dans tes erreurs :

- Pattern 1 — Moondream trop permissif (8 cas) : Cul noir, Chlorose, Aleurodes, Tétranyque, Piéride, Puceron, Pourriture blanche, Rouille, tous annotés F mais prédits entailment avec des confiances parfois très élevées (0.95 pour Puceron, 0.89 pour Pourriture blanche). Ce n'est pas une erreur de seuil, c'est un comportement systématique.
- Pattern 2 — GPT-5 trop conservateur (4 cas) : Cul noir, Chlorose, Mildiou, Pourriture blanche, tous annotés V mais prédits neutral. GPT-5 "voit" que la description est partiellement correcte mais pas suffisamment précise pour déclencher entailment. C'est sémantiquement plus honnête que Moondream.
- Pattern 3 — Les 2 carences (Cul noir + Chlorose) ratées par les deux modèles : accuracy Carence = 0.0 pour les deux. C'est la limite de domaine la plus nette — probablement parce que les carences ont une terminologie très spécifique (calcium, chlorophylle) peu représentée dans SNLI.
 ---
La conclusion finale sera : GPT-5 est le modèle plus fiable pour une pipeline de diagnostic phytopathologique, mais les deux modèles échouent sur les carences, ce qui suggère soit de re-finetuner sur ce domaine, soit d'utiliser un modèle spécialisé pour ce type.

# Phase 4 - Définition du seuil de qualité

Seuil et conclusion

Visualisation



## Seuil et conclusion

In [ ]:
print("\n ---- PHASE 4 : SEUIL DE QUALITÉ & CONCLUSION -------")

# ── CORRECTION : utiliser le corrected_score calculé en Phase 3 ──
# (pas de recalcul avec une autre formule)
print("\n 1. RAPPEL SCORES CORRIGÉS (Phase 3)")
print("-" * 40)
scores = df_results.groupby('model_source')['corrected_score'].agg(['mean','std'])
print(scores.round(3))
# Attendu : GPT-5 +0.285 | Moondream -0.215

# ── SEUIL OPÉRATIONNEL : prob_entailment >= 0.70 ──
print("\n 2. DÉFINITION DU SEUIL OPÉRATIONNEL")
print("-" * 40)
THRESHOLD = 0.70

print(f"\n Seuil retenu : prob_entailment >= {THRESHOLD}")
print("""
 Justification :
   - prob_entailment moyenne sur entailments = 0.695 → 0.70 est la médiane naturelle
   - Contradiction unique à 0.980 → toujours capturée
   - Évite les entailments "mous" (0.45-0.69) qui sont des faux positifs Moondream
""")

# Appliquer le seuil
df_results['decision'] = df_results.apply(lambda row: (
    'VALIDÉ'    if row['predicted_label'] == 'entailment'
                   and row['prob_entailment'] >= THRESHOLD
    else 'INCOMPLET' if row['predicted_label'] == 'neutral'
    else 'REJETÉ'   if row['predicted_label'] == 'contradiction'
    else 'INCERTAIN'  # entailment < seuil
), axis=1)

print("\n Distribution des décisions par modèle :")
decisions = df_results.groupby(['model_source','decision']).size().unstack(fill_value=0)
print(decisions)

# ── FIABILITÉ CORRIGÉE : haute confiance ET annotation correcte ──
print("\n 3. CAS FIABLES (confiance >= 0.70 ET prédiction correcte)")
print("-" * 40)

reliable_correct = df_results[
    (df_results['high_confidence'] == True) &
    (df_results['nli_correct'] == True)
]
print(f"\n Fiables ET corrects : {len(reliable_correct)} / {len(df_results)}")
print(reliable_correct[[
    'image','model_source','annotation','predicted_label',
    'confidence','corrected_score','nli_correct'
]].to_string(index=False))

# ── TABLEAU DE SYNTHÈSE FINAL ──
print("\n 4. TABLEAU DE SYNTHÈSE FINAL")
print("-" * 40)

summary = df_results.groupby('model_source').agg(
    accuracy        = ('nli_correct',      'mean'),
    corrected_score = ('corrected_score',  'mean'),
    conf_moyenne    = ('confidence',       'mean'),
    gap_moyen       = ('confidence_gap',   'mean'),
    nb_valide       = ('decision',         lambda x: (x == 'VALIDÉ').sum()),
    nb_incomplet    = ('decision',         lambda x: (x == 'INCOMPLET').sum()),
    nb_incertain    = ('decision',         lambda x: (x == 'INCERTAIN').sum()),
    nb_rejeté       = ('decision',         lambda x: (x == 'REJETÉ').sum()),
).round(3)

print(summary.T)  # Transposé pour lisibilité

# ── CONCLUSION ──
print("\n" + "="*60)
print(" CONCLUSION FINALE")
print("="*60)

print(f"""
Modèle recommandé : GPT-5
  → Accuracy  : {df_results[df_results['model_source']=='GPT-5']['nli_correct'].mean():.1%}
  → Score corrigé : {df_results[df_results['model_source']=='GPT-5']['corrected_score'].mean():+.3f}
  → Plus précis, moins de faux positifs, seule contradiction détectée (Tétranyque, conf=0.98)

Moondream :
  → Accuracy  : {df_results[df_results['model_source']=='Moondream']['nli_correct'].mean():.1%}
  → Score corrigé : {df_results[df_results['model_source']=='Moondream']['corrected_score'].mean():+.3f}
  → Biais de permissivité : prédit entailment même sur descriptions fausses

Seuil opérationnel retenu : prob_entailment >= {THRESHOLD}
  → En dessous : prédiction "INCERTAIN" — nécessite validation humaine
  → Contradiction >= 0.70 : rejet automatique de la description

Limite principale identifiée : Carences (accuracy = 0% pour les deux modèles)
  → Recommandation : finetuning sur données phytopathologiques spécialisées
    ou règles expertes complémentaires pour Carence/Chlorose
""")

##  VISUALISATION


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# ── PALETTE ──────────────────────────────────────────────
GREEN  = '#22c55e'; GREEN_DIM  = '#166534'; GREEN_BG  = '#0a2e1a'
AMBER  = '#f59e0b'; AMBER_DIM  = '#78350f'; AMBER_BG  = '#2a1f00'
RED    = '#ef4444'; RED_DIM    = '#7f1d1d'; RED_BG    = '#2a0a0a'
BLUE   = '#60a5fa'; BLUE_DIM   = '#1e3a5f'; BLUE_BG   = '#0a1a2e'
BG     = '#0f1117'; SURFACE    = '#181c27'; SURFACE2  = '#1f2436'
TEXT   = '#e2e8f0'; MUTED      = '#64748b'; BORDER    = '#2a3050'

fig = plt.figure(figsize=(20, 24), facecolor=BG)
fig.suptitle(
    'Pipeline IA  --  Diagnostic Phytopathologique  |  Rapport DJMA',
    fontsize=18, fontweight='bold', color='white',
    y=0.98, fontfamily='monospace'
)

gs = gridspec.GridSpec(
    4, 3,
    figure=fig,
    hspace=0.55, wspace=0.35,
    top=0.94, bottom=0.04,
    left=0.06, right=0.97
)

def style_ax(ax, title='', subtitle=''):
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values():
        spine.set_color(BORDER)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    if title:
        full_title = f'{title}\n{subtitle}' if subtitle else title
        ax.set_title(full_title, color=TEXT, fontsize=11, fontweight='bold',
                     pad=10, loc='left', fontfamily='monospace')

# ══════════════════════════════════════════════════════════
# 0) BANNER KPIs  (ligne 0, span 3 cols)
# ══════════════════════════════════════════════════════════
ax_banner = fig.add_subplot(gs[0, :])
ax_banner.set_facecolor(GREEN_BG)
for spine in ax_banner.spines.values():
    spine.set_color(GREEN_DIM)
ax_banner.set_xticks([]); ax_banner.set_yticks([])

kpis = [
    ('GPT-5\nrecommande',   'Accuracy 50%\nvs 33% Moondream',   GREEN),
    ('Score qualite\ncorrige', 'GPT-5 : +0.285\nMoondream : -0.215', AMBER),
    ('Cas fiables\ndetectes',  '5 / 24 paires\n(conf >= 0.70)',      BLUE),
    ('Contradiction\nrobuste', 'Tetranyque GPT-5\nconf = 0.98',       RED),
    ('Limite\nidentifiee',   'Carences : 0%\npour les 2 modeles',   AMBER),
]

for i, (title, val, color) in enumerate(kpis):
    x = 0.1 + i * 0.2
    ax_banner.text(x, 0.72, title, transform=ax_banner.transAxes,
                   ha='center', va='center', color=color,
                   fontsize=10, fontweight='bold', fontfamily='monospace')
    ax_banner.text(x, 0.28, val, transform=ax_banner.transAxes,
                   ha='center', va='center', color=TEXT,
                   fontsize=9, linespacing=1.5)
    if i < len(kpis)-1:
        ax_banner.axvline(x=0.2*(i+1), color=BORDER, lw=1)

ax_banner.set_title('  SYNTHESE EXECUTIVE', color=GREEN,
                     fontsize=12, fontweight='bold', loc='left',
                     pad=8, fontfamily='monospace')

# ══════════════════════════════════════════════════════════
# 1) ACCURACY  (ligne 1, col 0)
# ══════════════════════════════════════════════════════════
ax1 = fig.add_subplot(gs[1, 0])
style_ax(ax1, 'ACCURACY', 'predictions correctes vs annotations')

models = ['GPT-5', 'Moondream']
accs   = [50, 33.3]
colors = [GREEN, AMBER]
bars = ax1.bar(models, accs, color=colors, width=0.5,
               edgecolor=BORDER, linewidth=0.8)

for bar, val in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f'{val:.0f}%', ha='center', va='bottom',
             color=TEXT, fontsize=14, fontweight='bold')

ax1.axhline(y=50, color=MUTED, lw=1, ls='--', alpha=0.5)
ax1.text(1.45, 51.5, 'max atteint', color=MUTED, fontsize=8)
ax1.set_ylim(0, 70)
ax1.set_ylabel('Accuracy (%)', color=MUTED, fontsize=9)
ax1.set_facecolor(SURFACE)

# ══════════════════════════════════════════════════════════
# 2) SCORE CORRIGE  (ligne 1, col 1)
# ══════════════════════════════════════════════════════════
ax2 = fig.add_subplot(gs[1, 1])
style_ax(ax2, 'SCORE QUALITE CORRIGE', '-1 = mauvais  /  0 = neutre  /  +1 = parfait')

scores    = [0.285, -0.215]
colors_sc = [GREEN, RED]
bars2 = ax2.bar(models, scores, color=colors_sc, width=0.5,
                edgecolor=BORDER, linewidth=0.8)
ax2.axhline(y=0, color=TEXT, lw=1.2, alpha=0.6)

for bar, val in zip(bars2, scores):
    ypos = val + 0.015 if val >= 0 else val - 0.035
    ax2.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{val:+.3f}', ha='center', va='bottom',
             color=TEXT, fontsize=13, fontweight='bold')

ax2.set_ylim(-0.45, 0.50)
ax2.set_ylabel('Score moyen', color=MUTED, fontsize=9)
ax2.set_facecolor(SURFACE)

ax2.annotate('Biais de\npermissivite', xy=(1, -0.215),
             xytext=(1.25, -0.35), color=RED, fontsize=8,
             arrowprops=dict(arrowstyle='->', color=RED, lw=1.2))

# ══════════════════════════════════════════════════════════
# 3) DISTRIBUTION DECISIONS  (ligne 1, col 2)
# ══════════════════════════════════════════════════════════
ax3 = fig.add_subplot(gs[1, 2])
style_ax(ax3, 'DECISIONS AUTOMATIQUES', 'par modele  |  seuil prob >= 0.70')

categories = ['VALIDE', 'INCOMPLET', 'INCERTAIN', 'REJETE']
gpt5_vals  = [3, 5, 3, 1]
moon_vals  = [5, 2, 5, 0]
c_dec      = [GREEN, AMBER, BLUE, RED]

x3 = np.arange(len(categories))
w  = 0.35

bars_g = ax3.bar(x3 - w/2, gpt5_vals, w, label='GPT-5',
                  color=[c+'99' for c in c_dec], edgecolor=c_dec, linewidth=1.5)
bars_m = ax3.bar(x3 + w/2, moon_vals, w, label='Moondream',
                  color=[c+'44' for c in c_dec], edgecolor=c_dec, linewidth=1.5,
                  linestyle='--')

ax3.set_xticks(x3)
ax3.set_xticklabels(categories, fontsize=8, color=MUTED)
ax3.set_ylabel('Nombre de paires', color=MUTED, fontsize=9)
ax3.legend(facecolor=SURFACE2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax3.set_ylim(0, 7.5)
ax3.set_facecolor(SURFACE)

for bar in list(bars_g) + list(bars_m):
    h = bar.get_height()
    if h > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, h + 0.1,
                 str(int(h)), ha='center', va='bottom', color=TEXT, fontsize=9)

# ══════════════════════════════════════════════════════════
# 4) PERFORMANCE PAR TYPE  (ligne 2, col 0)
# ══════════════════════════════════════════════════════════
ax4 = fig.add_subplot(gs[2, 0])
style_ax(ax4, 'PERFORMANCE PAR TYPE', 'accuracy Moondream vs GPT-5')

types     = ['Maladies\n(5)', 'Ravageurs\n(5)', 'Carences\n(2)']
gpt5_type = [60, 60,  0]
moon_type = [60, 20,  0]
x4 = np.arange(len(types))

ax4.bar(x4 - 0.2, gpt5_type, 0.35, label='GPT-5',
        color=GREEN, alpha=0.85, edgecolor=GREEN_DIM)
ax4.bar(x4 + 0.2, moon_type, 0.35, label='Moondream',
        color=AMBER, alpha=0.85, edgecolor=AMBER_DIM)

for i, (g, m) in enumerate(zip(gpt5_type, moon_type)):
    ax4.text(i-0.2, g+2, f'{g}%', ha='center', color=TEXT, fontsize=9, fontweight='bold')
    ax4.text(i+0.2, m+2, f'{m}%', ha='center', color=TEXT, fontsize=9, fontweight='bold')

ax4.axhspan(0, 5, color=RED, alpha=0.08)
ax4.text(2.3, 2, 'Limite critique', color=RED, fontsize=8, va='center')
ax4.set_xticks(x4); ax4.set_xticklabels(types, fontsize=9, color=MUTED)
ax4.set_ylim(0, 80); ax4.set_ylabel('Accuracy (%)', color=MUTED, fontsize=9)
ax4.legend(facecolor=SURFACE2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax4.set_facecolor(SURFACE)

# ══════════════════════════════════════════════════════════
# 5) DISTRIBUTION CONFIANCE  (ligne 2, col 1)
# ══════════════════════════════════════════════════════════
ax5 = fig.add_subplot(gs[2, 1])
style_ax(ax5, 'DISTRIBUTION CONFIANCE', 'prob de la classe predite  |  24 paires')

conf_gpt5 = [0.664, 0.623, 0.842, 0.980, 0.701, 0.710, 0.684,
             0.664, 0.583, 0.593, 0.720, 0.534]
conf_moon = [0.538, 0.677, 0.922, 0.575, 0.649, 0.953, 0.759,
             0.477, 0.745, 0.894, 0.783, 0.451]

ax5.hist(conf_gpt5, bins=8, alpha=0.7, color=BLUE,  label='GPT-5',
         edgecolor=BORDER, range=(0.4, 1.0))
ax5.hist(conf_moon, bins=8, alpha=0.7, color=AMBER, label='Moondream',
         edgecolor=BORDER, range=(0.4, 1.0))

ax5.axvline(x=0.70, color=GREEN, lw=2, ls='--', label='Seuil 0.70')
ax5.text(0.71, 3.6, 'seuil', color=GREEN, fontsize=8, rotation=90, va='top')

ax5.set_xlabel('Confiance (probabilite)', color=MUTED, fontsize=9)
ax5.set_ylabel('Nb de paires', color=MUTED, fontsize=9)
ax5.legend(facecolor=SURFACE2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax5.set_facecolor(SURFACE)

# ══════════════════════════════════════════════════════════
# 6) BIAIS PERMISSIVITE  (ligne 2, col 2)
# ══════════════════════════════════════════════════════════
ax6 = fig.add_subplot(gs[2, 2])
style_ax(ax6, 'BIAIS DE PERMISSIVITE', 'entailments incorrects sur cas annotes FAUX')

labels6  = ['Moondream', 'GPT-5']
faux_ent = [8, 2]   # faux positifs (entailment sur ann=F)
faux_ok  = [1, 4]   # neutral/contradiction corrects sur ann=F
totaux   = [9, 6]

x6 = np.arange(len(labels6))
ax6.bar(x6, faux_ent, color=RED,   alpha=0.85, label='Faux positifs (entailment sur F)',
        edgecolor=RED_DIM)
ax6.bar(x6, faux_ok,  bottom=faux_ent, color=GREEN, alpha=0.85,
        label='Bien predits (neutral/contradiction)', edgecolor=GREEN_DIM)

for i, (fe, fo, tot) in enumerate(zip(faux_ent, faux_ok, totaux)):
    ax6.text(i, fe/2, f'{fe}/{tot}\nfaux+',
             ha='center', va='center', color='white', fontsize=10, fontweight='bold')
    if fo > 0:
        ax6.text(i, fe + fo/2, f'{fo}/{tot}\ncorrects',
                 ha='center', va='center', color='white', fontsize=9)

ax6.set_xticks(x6)
ax6.set_xticklabels(labels6, fontsize=10, color=TEXT, fontweight='bold')
ax6.set_ylabel('Nb de paires (ann=Faux)', color=MUTED, fontsize=9)
ax6.set_ylim(0, 12)
ax6.legend(facecolor=SURFACE2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8,
           loc='upper right')
ax6.set_facecolor(SURFACE)

# ══════════════════════════════════════════════════════════
# 7) TABLEAU CAS FIABLES  (ligne 3, span 2 cols)
# ══════════════════════════════════════════════════════════
ax7 = fig.add_subplot(gs[3, :2])
ax7.set_facecolor(SURFACE)
for spine in ax7.spines.values(): spine.set_color(BORDER)
ax7.set_xticks([]); ax7.set_yticks([])
ax7.set_title(
    '  CAS FIABLES  --  confiance >= 0.70 + prediction correcte  (5/24)',
    color=TEXT, fontsize=11, fontweight='bold', loc='left',
    pad=8, fontfamily='monospace'
)

cols = ['Pathologie', 'Modele', 'Label NLI', 'Confiance', 'Score corrige', 'Annotation']
rows = [
    ['Aleurodes',        'GPT-5',     'neutral',       '0.842', '+0.765', 'Faux  OK'],
    ['Tetranyque',       'GPT-5',     'contradiction', '0.980', '+0.984', 'Faux  OK'],
    ['Tuta absoluta',    'Moondream', 'neutral',       '0.759', '+0.525', 'Faux  OK'],
    ['Pourriture grise', 'Moondream', 'entailment',    '0.745', '+0.621', 'Vrai  OK'],
    ['Rouille',          'GPT-5',     'entailment',    '0.720', '+0.707', 'Vrai  OK'],
]

label_colors = {'entailment': GREEN, 'neutral': AMBER, 'contradiction': RED}

table = ax7.table(
    cellText=rows, colLabels=cols,
    cellLoc='center', loc='center',
    bbox=[0.01, 0.05, 0.98, 0.88]
)
table.auto_set_font_size(False)
table.set_fontsize(10)

for (r, c), cell in table.get_celld().items():
    cell.set_facecolor(SURFACE)
    cell.set_edgecolor(BORDER)
    cell.set_text_props(color=TEXT)
    if r == 0:
        cell.set_facecolor(SURFACE2)
        cell.set_text_props(color=MUTED, fontweight='bold', fontsize=9)
    if r > 0 and c == 2:
        lbl = rows[r-1][2]
        cell.set_text_props(color=label_colors.get(lbl, TEXT), fontweight='bold')
    if r > 0 and c == 0:
        cell.set_facecolor(SURFACE2)

# ══════════════════════════════════════════════════════════
# 8) RECOMMANDATIONS  (ligne 3, col 2)
# ══════════════════════════════════════════════════════════
ax8 = fig.add_subplot(gs[3, 2])
ax8.set_facecolor(SURFACE)
for spine in ax8.spines.values(): spine.set_color(BORDER)
ax8.set_xticks([]); ax8.set_yticks([])
ax8.set_title('  RECOMMANDATIONS', color=BLUE,
              fontsize=11, fontweight='bold', loc='left',
              pad=8, fontfamily='monospace')

recos = [
    ('1',  GREEN, '-> Amélioration du modèle NLI Finetuner DistilBERT+LoRA\n sur un corpus phytopathologique(carence)'),
    ('2',  AMBER, '-> Gérer le problème de troncature avec un max_length plus grand'),
    ('3',   BLUE,  '-> Augmenter la taille de la base inference, surtout les carences\n necessite des donnees supplementaires'),
]

y_pos = 0.87
for horizon, color, text in recos:
    ax8.text(0.04, y_pos, f'[{horizon}]', transform=ax8.transAxes,
             color=color, fontsize=9, fontweight='bold', va='top',
             fontfamily='monospace')
    ax8.text(0.04, y_pos - 0.07, text, transform=ax8.transAxes,
             color=TEXT, fontsize=9, va='top', linespacing=1.5)

    # FIX : ligne de separation via plot (pas axhline)
    y_data = y_pos - 0.22
    ax8.plot([0.03, 0.97], [y_data, y_data],
             color=BORDER, lw=0.8, transform=ax8.transAxes, clip_on=False)
    y_pos -= 0.26

plt.savefig(
    '/content/drive/MyDrive/PROJET_DJMA/djma_dashboard_final.png',
    dpi=180, bbox_inches='tight', facecolor=BG
)
plt.show()
print("Dashboard sauvegarde : djma_dashboard_final.png")

---